# PlatesV2 — Cleaned vs Dirty plate classification**Задача:** бинарная классификация изображений тарелок (cleaned/dirty) для Kaggle-соревнования [platesv2](https://www.kaggle.com/c/platesv2).**Особенность:** датасет крошечный — **40 train** vs **744 test**. Это few-shot задача, требующая transfer learning + сильных аугментаций + кросс-валидации.**Финальный подход:** ResNet18 (ImageNet pretrained, backbone заморожен) + 5-fold StratifiedKFold CV + TTA(x6) ensemble.**Public LB:** 0.83467 (accuracy).

## 1. Импорты и конфигурация

In [ ]:
from __future__ import annotationsimport jsonimport randomfrom pathlib import Pathimport matplotlib.pyplot as pltimport numpy as npimport pandas as pdimport torchimport torch.nn as nnimport torch.optim as optimfrom PIL import Imagefrom sklearn.metrics import (    accuracy_score,    classification_report,    confusion_matrix,    f1_score,)from sklearn.model_selection import StratifiedKFoldfrom torch.utils.data import DataLoader, Datasetfrom torchvision import models, transformsfrom tqdm import tqdmPROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()DATA_DIR = PROJECT_ROOT / "data" / "plates"OUTPUTS_DIR = PROJECT_ROOT / "outputs"OUTPUTS_DIR.mkdir(exist_ok=True)SEED = 42IMG_SIZE = 224BATCH_SIZE = 8NUM_EPOCHS = 25LR_HEAD = 1e-3LR_BACKBONE = 1e-4WEIGHT_DECAY = 1e-4N_FOLDS = 5DROPOUT = 0.3CLASSES = ["cleaned", "dirty"]CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}IDX_TO_CLASS = {i: c for c, i in CLASS_TO_IDX.items()}IMAGENET_MEAN = [0.485, 0.456, 0.406]IMAGENET_STD = [0.229, 0.224, 0.225]def set_seed(seed: int) -> None:    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)    torch.cuda.manual_seed_all(seed)    torch.backends.cudnn.deterministic = True    torch.backends.cudnn.benchmark = Falseset_seed(SEED)device = torch.device("cuda" if torch.cuda.is_available() else "cpu")print(f"Device: {device}")

## 2. Загрузка данныхСтруктура датасета:- `data/plates/train/cleaned/` — 20 фото чистых тарелок- `data/plates/train/dirty/` — 20 фото грязных тарелок- `data/plates/test/` — 744 фото без меток**Train в 18 раз меньше test** — это ключевая особенность задачи.

In [ ]:
def collect_train_files() -> pd.DataFrame:    rows = []    for cls in CLASSES:        for p in sorted((DATA_DIR / "train" / cls).iterdir()):            if p.suffix.lower() in {".jpg", ".jpeg", ".png"}:                rows.append({"path": str(p), "class": cls, "label": CLASS_TO_IDX[cls]})    return pd.DataFrame(rows)def collect_test_files() -> pd.DataFrame:    rows = []    for p in sorted((DATA_DIR / "test").iterdir()):        if p.suffix.lower() in {".jpg", ".jpeg", ".png"}:            rows.append({"path": str(p), "id": p.stem})    return pd.DataFrame(rows)train_df = collect_train_files().reset_index(drop=True)test_df = collect_test_files().reset_index(drop=True)print(f"Train: {len(train_df)} | Test: {len(test_df)}")print(train_df['class'].value_counts())

## 3. Разведывательный анализ (EDA)Смотрим:1. Баланс классов (важно для выбора метрики и стратификации)2. Распределение размеров изображений (нужно для выбора `IMG_SIZE`)3. Примеры из каждого класса (нужно для выбора аугментаций)

In [ ]:
sizes = [Image.open(p).size for p in train_df["path"]]widths, heights = zip(*sizes)fig, axes = plt.subplots(1, 3, figsize=(15, 4))train_df["class"].value_counts().plot.bar(ax=axes[0], color=["steelblue", "tomato"])axes[0].set_title("Class balance (train)")axes[1].hist(widths, bins=15, color="steelblue", alpha=0.7, label="width")axes[1].hist(heights, bins=15, color="tomato", alpha=0.7, label="height")axes[1].set_title("Image dimensions"); axes[1].legend()axes[2].scatter(widths, heights, alpha=0.6)axes[2].set_xlabel("width"); axes[2].set_ylabel("height"); axes[2].set_title("W x H")plt.tight_layout(); plt.savefig(OUTPUTS_DIR / "eda_distributions.png", dpi=100); plt.show()print(f"Width range: {min(widths)}-{max(widths)}, mean={np.mean(widths):.0f}")print(f"Height range: {min(heights)}-{max(heights)}, mean={np.mean(heights):.0f}")

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(15, 6))for row, cls in enumerate(CLASSES):    sample_paths = train_df[train_df["class"] == cls]["path"].sample(5, random_state=SEED).tolist()    for col, p in enumerate(sample_paths):        axes[row, col].imshow(Image.open(p))        axes[row, col].set_title(cls); axes[row, col].axis("off")plt.tight_layout(); plt.savefig(OUTPUTS_DIR / "eda_samples.png", dpi=100); plt.show()

**Выводы EDA:**- Классы строго сбалансированы (20/20) — можно использовать `accuracy` как основную метрику.- Размеры варьируются 256-455 px, разное соотношение сторон → нужен `Resize` к фиксированному квадрату.- Тарелки разной формы, цвета, фона, освещения. Грязь визуально разная (остатки еды, пятна, налёт).- 40 картинок мало → critical: transfer learning + aggressive augmentations + cross-validation.

## 4. Препроцессинг и аугментации**Train transform** — сильные аугментации, искусственно раздувают 40 фото в условные тысячи:- `RandomCrop` после `Resize(256)` — модель не привязана к точному кадрированию- `Horizontal/Vertical Flip` — тарелку можно крутить как угодно- `RandomRotation(30°)` — разные углы съёмки- `ColorJitter` — разное освещение/баланс белого- `RandomPerspective` — лёгкая 3D-деформация- `Normalize` под ImageNet stats — обязательно для предобученной модели**Eval transform** — только `Resize` + `Normalize` (детерминированный).

In [ ]:
def build_transforms():    train_tf = transforms.Compose([        transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),        transforms.RandomCrop(IMG_SIZE),        transforms.RandomHorizontalFlip(p=0.5),        transforms.RandomVerticalFlip(p=0.5),        transforms.RandomRotation(degrees=30),        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05),        transforms.RandomPerspective(distortion_scale=0.2, p=0.3),        transforms.ToTensor(),        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),    ])    eval_tf = transforms.Compose([        transforms.Resize((IMG_SIZE, IMG_SIZE)),        transforms.ToTensor(),        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),    ])    return train_tf, eval_tfclass PlatesDataset(Dataset):    def __init__(self, df, transform, has_labels=True):        self.paths = df["path"].tolist()        self.transform = transform        self.has_labels = has_labels        self.labels = df["label"].tolist() if has_labels else None    def __len__(self): return len(self.paths)    def __getitem__(self, idx):        img = Image.open(self.paths[idx]).convert("RGB")        img = self.transform(img)        if self.has_labels: return img, self.labels[idx]        return img, self.paths[idx]train_tf, eval_tf = build_transforms()

## 5. Модель — ResNet18 transfer learning**Выбор архитектуры:**- Пробовали ResNet50 + разморозка `layer4` → переобучение (train acc → 1.0, public LB упал с 0.83 до 0.72).- Финальный выбор — **ResNet18 с полностью замороженным backbone**, тренируется только новая голова (Dropout + Linear на 2 класса).- Меньше параметров = меньше переобучения на 32 картинках за фолд.**Почему transfer learning обязателен:**40 картинок недостаточно для обучения CNN с нуля. ImageNet-pretrained веса уже умеют видеть края/текстуры/формы — нам нужно только переучить классификатор.

In [ ]:
def build_model() -> nn.Module:    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)    for p in model.parameters():        p.requires_grad = False  # backbone заморожен    in_features = model.fc.in_features    model.fc = nn.Sequential(        nn.Dropout(DROPOUT),        nn.Linear(in_features, 2),    )    return modelmodel = build_model()n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)n_total = sum(p.numel() for p in model.parameters())print(f"Trainable: {n_trainable:,} / Total: {n_total:,}")

## 6. Цикл обучения с регуляризацией**Гиперпараметры:**- Optimizer: **Adam(lr=1e-3, weight_decay=1e-4)** — стандарт для transfer learning- Loss: `CrossEntropyLoss` (бинарная классификация через 2 logits)- Scheduler: `ReduceLROnPlateau(factor=0.5, patience=3)` — уменьшает LR когда val метрика не растёт- `Dropout(0.3)` в голове — регуляризация- 25 эпох — этого хватает с замороженным backbone (тренируется только FC слой)

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):    model.train()    losses, preds, targets = [], [], []    for x, y in loader:        x, y = x.to(device), y.to(device)        optimizer.zero_grad()        out = model(x)        loss = criterion(out, y)        loss.backward(); optimizer.step()        losses.append(loss.item())        preds.extend(out.argmax(1).cpu().numpy()); targets.extend(y.cpu().numpy())    return float(np.mean(losses)), accuracy_score(targets, preds)@torch.no_grad()def evaluate(model, loader, criterion, device):    model.eval()    losses, preds, targets = [], [], []    for x, y in loader:        x, y = x.to(device), y.to(device)        out = model(x); loss = criterion(out, y)        losses.append(loss.item())        preds.extend(out.argmax(1).cpu().numpy()); targets.extend(y.cpu().numpy())    return float(np.mean(losses)), accuracy_score(targets, preds), preds, targetsdef train_model(model, train_loader, val_loader, device, num_epochs):    criterion = nn.CrossEntropyLoss()    trainable = [p for p in model.parameters() if p.requires_grad]    optimizer = optim.Adam(trainable, lr=LR_HEAD, weight_decay=WEIGHT_DECAY)    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=3)    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}    best_val_acc, best_state = 0.0, None    for epoch in range(1, num_epochs + 1):        tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)        val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion, device)        scheduler.step(val_acc)        history["train_loss"].append(tr_loss); history["train_acc"].append(tr_acc)        history["val_loss"].append(val_loss); history["val_acc"].append(val_acc)        if val_acc > best_val_acc:            best_val_acc = val_acc            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}    if best_state is not None:        model.load_state_dict(best_state)    return model, history, best_val_acc

## 7. 5-Fold Cross-Validation**Почему K-fold обязателен:**- Один train/val split при 40 картинках = val=8. Метрика accuracy на 8 примерах дискретна (0.5, 0.625, 0.75, 0.875) → нельзя надёжно сравнивать модели.- 5-fold = 5 моделей с разными train/val подмножествами → усреднение даёт стабильную OOF оценку.- Усреднение предсказаний 5 моделей на test = ансамбль = +1-2% к итоговому score.

In [ ]:
@torch.no_grad()def predict_with_tta(model, test_df, eval_tf, device, n_tta=5):    """Test-time augmentation: усредняем предсказания по n_tta аугментированным версиям + 1 базовая."""    tta_tf = transforms.Compose([        transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),        transforms.RandomCrop(IMG_SIZE),        transforms.RandomHorizontalFlip(p=0.5),        transforms.RandomVerticalFlip(p=0.5),        transforms.RandomRotation(degrees=20),        transforms.ToTensor(),        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),    ])    model.eval()    probs_sum = np.zeros((len(test_df), 2), dtype=np.float32)    # базовый прогон    base_ds = PlatesDataset(test_df, eval_tf, has_labels=False)    base_loader = DataLoader(base_ds, batch_size=BATCH_SIZE, shuffle=False)    base_probs = []    for x, _ in base_loader:        x = x.to(device)        base_probs.append(torch.softmax(model(x), dim=1).cpu().numpy())    probs_sum += np.vstack(base_probs)    # TTA проходы    for _ in range(n_tta):        ds = PlatesDataset(test_df, tta_tf, has_labels=False)        loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False)        probs = []        for x, _ in loader:            x = x.to(device)            probs.append(torch.softmax(model(x), dim=1).cpu().numpy())        probs_sum += np.vstack(probs)    return probs_sum / (n_tta + 1)

In [ ]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)fold_val_accs, fold_histories = [], []test_probs_sum = np.zeros((len(test_df), 2), dtype=np.float32)oof_preds = np.zeros(len(train_df), dtype=np.int64)oof_true = train_df["label"].valuesfor fold, (train_idx, val_idx) in enumerate(skf.split(train_df, train_df["label"]), start=1):    print(f"FOLD {fold}/{N_FOLDS}")    tr = train_df.iloc[train_idx].reset_index(drop=True)    vl = train_df.iloc[val_idx].reset_index(drop=True)    train_ds = PlatesDataset(tr, train_tf); val_ds = PlatesDataset(vl, eval_tf)    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)    model = build_model().to(device)    model, hist, best_acc = train_model(model, train_loader, val_loader, device, NUM_EPOCHS)    fold_val_accs.append(best_acc); fold_histories.append(hist)    _, _, vp, _ = evaluate(model, val_loader, nn.CrossEntropyLoss(), device)    oof_preds[val_idx] = vp    test_probs_sum += predict_with_tta(model, test_df, eval_tf, device, n_tta=5)    torch.save(model.state_dict(), OUTPUTS_DIR / f"model_resnet18_fold{fold}.pt")test_probs = test_probs_sum / N_FOLDSprint(f"Fold val accs: {[f'{a:.3f}' for a in fold_val_accs]}")print(f"Mean: {np.mean(fold_val_accs):.3f} +- {np.std(fold_val_accs):.3f}")

## 8. Графики обучения

In [ ]:
hist = fold_histories[0]fig, axes = plt.subplots(1, 2, figsize=(12, 4))axes[0].plot(hist["train_loss"], label="train"); axes[0].plot(hist["val_loss"], label="val")axes[0].set_title("Loss"); axes[0].set_xlabel("epoch"); axes[0].legend()axes[1].plot(hist["train_acc"], label="train"); axes[1].plot(hist["val_acc"], label="val")axes[1].set_title("Accuracy"); axes[1].set_xlabel("epoch"); axes[1].legend()plt.tight_layout(); plt.savefig(OUTPUTS_DIR / "training_history.png", dpi=100); plt.show()

## 9. Оценка модели**OOF (Out-Of-Fold)** — предсказания на val-фолдах склеены: каждая train-картинка получает предсказание от модели, которая её не видела. Это самая честная оценка качества на train-данных.

In [ ]:
oof_acc = accuracy_score(oof_true, oof_preds)oof_f1 = f1_score(oof_true, oof_preds, average="macro")print(f"OOF accuracy: {oof_acc:.3f}, macro-F1: {oof_f1:.3f}")print(classification_report(oof_true, oof_preds, target_names=CLASSES, zero_division=0))cm = confusion_matrix(oof_true, oof_preds, labels=[0, 1])fig, ax = plt.subplots(figsize=(5, 4))ax.imshow(cm, cmap="Blues")for i in range(2):    for j in range(2):        ax.text(j, i, cm[i, j], ha="center", va="center")ax.set_xticks([0, 1]); ax.set_xticklabels(CLASSES)ax.set_yticks([0, 1]); ax.set_yticklabels(CLASSES)ax.set_xlabel("Predicted"); ax.set_ylabel("True")ax.set_title(f"OOF confusion matrix (acc={oof_acc:.2f})")plt.tight_layout(); plt.savefig(OUTPUTS_DIR / "confusion_matrix.png", dpi=100); plt.show()

## 10. Анализ ошибок

In [ ]:
err_mask = oof_preds != oof_trueerr_df = train_df.iloc[np.where(err_mask)[0]].reset_index(drop=True)err_df["pred"] = oof_preds[err_mask]; err_df["true"] = oof_true[err_mask]print(f"OOF errors: {len(err_df)} / {len(train_df)}")n_show = min(8, len(err_df))fig, axes = plt.subplots(1, n_show, figsize=(3 * n_show, 3))for ax, (_, row) in zip(axes, err_df.head(n_show).iterrows()):    ax.imshow(Image.open(row["path"]))    ax.set_title(f"T:{IDX_TO_CLASS[row['true']]}\nP:{IDX_TO_CLASS[row['pred']]}")    ax.axis("off")plt.tight_layout(); plt.savefig(OUTPUTS_DIR / "validation_errors.png", dpi=100); plt.show()

**Типичные ошибки:**- Чистые тарелки с тенями/бликами модель путает с грязными.- Грязные тарелки на которых остатки еды мало контрастны с цветом тарелки.- Тарелки нестандартной формы (стеклянные, прозрачные) — мало похожи на train-примеры.

## 11. SubmissionФормируем CSV в формате конкурса: `id,label` где `label` — строка `cleaned`/`dirty`.

In [ ]:
pred_idx = test_probs.argmax(axis=1)pred_labels = [IDX_TO_CLASS[i] for i in pred_idx]sub = pd.DataFrame({"id": test_df["id"].values, "label": pred_labels}).sort_values("id").reset_index(drop=True)sub_path = OUTPUTS_DIR / "submission.csv"sub.to_csv(sub_path, index=False)print(f"Saved: {sub_path}")print(sub["label"].value_counts())sub.head(10)

## 12. Итоги и выводы| Метрика | Значение ||---|---|| OOF accuracy (5-fold) | ~0.68-0.75 || OOF macro F1 | ~0.67 || **Kaggle public LB** | **0.83467** |**Что сработало:**- Transfer learning с замороженным backbone (минимум параметров → минимум переобучения).- 5-fold cross-validation + усреднение предсказаний (вместо одного val split на 8 картинках).- TTA на тесте даёт +1-2%.**Что не сработало (исключено из финального решения):**- ResNet50: при 32 train-картинках слишком много параметров → нестабильно (фолд может уйти в 0.375).- Разморозка `layer4`: переобучение, train acc=1.0, public LB падает с 0.83 до 0.72.- Ансамбль ResNet18+ResNet50: разваливается из-за нестабильности ResNet50.- Усиленные аугментации (RandomErasing, Grayscale): на 40 картинках слишком разрушают сигнал.**Что попробовать для дальнейшего улучшения:**1. **GrabCut** для сегментации тарелки от фона + аугментация фона (даёт +5-7% по решениям топ-3 на каггле).2. **EfficientNet-B0/B1** — компромисс между ResNet18 (мало capacity) и ResNet50 (много).3. **MixUp / CutMix** аугментации.4. **Self-supervised pre-training** на test-картинках (744 шт. без меток) перед классификацией.5. **Pseudo-labeling**: уверенные test-предсказания добавить в train и переобучить.